# View Purge Behavior — `purge-view-metadata-on-drop`

> **New here? Read this first.** This notebook answers a precise question a manager
> raised: *what happens when you drop a VIEW under each value of the catalog config
> `polaris.config.purge-view-metadata-on-drop`?*

## Background

A view holds no data files — only a small `.gz.metadata.json`. Two catalog configs
interact when you drop one:

```
purge-view-metadata-on-drop = true   → drop tries to delete the view's metadata file
drop-with-purge.enabled              → gates whether purge is allowed at all
```

These two are **coupled**: with `purge-view-metadata-on-drop=true` AND
`drop-with-purge.enabled=false`, every view drop returns **403**. And even when the
drop succeeds (`204`), the metadata file is **not actually deleted** in our no-STS
environment — the same async-cleanup-task failure that affects table purge
(see `polaris-purge-test-report.md`, Apache Polaris #379).

## What this notebook proves

```
Case 1   purge-view-metadata=true  + drop-with-purge=false → 403 (cannot drop)
Case 2   purge-view-metadata=true  + drop-with-purge=true  → 204, file orphaned
Case 3   purge-view-metadata=false                         → 204, metadata-only
Truth table assembled from observed results.
```

## Prerequisites & how to run

```
Files:   polaris_test_utils.py, minio_rest.py, config/  (same folder)
Packages: pip install requests pyyaml opensearch-py pyarrow
Run:     run the setup cell (prints 🟢 DEV / 🔴 PROD), then top to bottom.
Switch:  change init_env("dev") → init_env("prod") in setup, re-run that cell.

⚠️  Storage access via MinIO REST API (minio_rest.py) — NO s3fs. No hardcoded creds.
    IMPORTANT for view files: the real metadata is a ~380-byte .gz.metadata.json
    one level INSIDE metadata/. We list recursively and ignore 0-byte directory
    markers (mc.list does both by default).
```

---


In [5]:
# ── Setup ─────────────────────────────────────────────────────────────────
# Environment switch HERE. Banner shows 🟢 DEV / 🔴 PROD. Change "dev" → "prod"
# and re-run THIS cell to switch (no kernel restart needed).
from polaris_test_utils import *
init_env("dev")

import uuid, time
tok = root_token()

TEST_CATALOG_V = "view-purge-test"
NS = "vp-ns"

def view_files(view_name):
    """Recursive list of REAL (>0 byte) files under a view's location."""
    return mc.list(f"{TEST_CATALOG_V}/{NS}/{view_name}/")   # mc.list filters 0-byte markers

def create_view(catalog, ns, name):
    return requests.post(f"{BASE_CAT}/{catalog}/namespaces/{ns}/views", headers=h(tok),
        json={"name": name, "default-namespace": [ns],
              "schema": {"type":"struct","schema-id":0,
                         "fields":[{"id":1,"name":"id","type":"long","required":True}]},
              "view-version": {"version-id":1,"timestamp-ms":int(time.time()*1000),"schema-id":0,
                               "default-namespace":[ns],"summary":{"engine":"spark"},
                               "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
              "location": f"s3a://{BUCKET}/{TEST_CATALOG_V}/{ns}/{name}/"})

def set_catalog_config(catalog, **flags):
    cat = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json()
    merged = {**cat.get("properties", {}), **{f"polaris.config.{k.replace('_','-')}":v for k,v in flags.items()}}
    requests.put(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})

print("✅ setup ready")


🟢 DEV   POLARIS_ENV=dev   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ setup ready


In [6]:
# Cleanup any leftover, then create a fresh catalog + namespace
print("Cleanup leftover...")
cats = [c["name"] for c in requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])]
if TEST_CATALOG_V in cats:
    set_catalog_config(TEST_CATALOG_V, drop_with_purge_enabled="true", purge_view_metadata_on_drop="false")
    grant_privilege(catalog=TEST_CATALOG_V, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)
    for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces", headers=h(tok)).json().get("namespaces", []):
        nsn = ns[0] if isinstance(ns, list) else ns
        for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG_V}?purgeRequested=true", headers=h(tok))
mc.delete_prefix(f"{TEST_CATALOG_V}/")

# Create catalog with purge-view-metadata-on-drop = TRUE (the default we're testing)
create_catalog(TEST_CATALOG_V, token=tok, properties={"polaris.config.purge-view-metadata-on-drop":"true"})
grant_privilege(catalog=TEST_CATALOG_V, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)
create_namespace(catalog=TEST_CATALOG_V, ns=NS, token=tok)
print("✅ catalog + namespace ready (purge-view-metadata-on-drop=true)")


Cleanup leftover...
✅ catalog + namespace ready (purge-view-metadata-on-drop=true)


## Case 1 — `purge-view-metadata=true` + `drop-with-purge=false`

**Why:** this is the *default* combination a user would hit. We expect a **403** —
the view drop is treated as a purge (because metadata-purge is on), but purge is
disabled, so it's blocked. This is the safety feature working, not a malfunction.


In [7]:
set_catalog_config(TEST_CATALOG_V, purge_view_metadata_on_drop="true", drop_with_purge_enabled="false")
V1 = "view-true-nopurge"
create_view(TEST_CATALOG_V, NS, V1)
print(f"Created {V1}")

r = requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{NS}/views/{V1}", headers=h(tok))
print(f"Drop status: {r.status_code}")
if r.status_code == 403:
    print("✅ Case 1: 403 — view CANNOT be dropped (metadata-purge=true needs drop-with-purge=true)")
else:
    print(f"ℹ️ unexpected: {r.text[:160]}")


Created view-true-nopurge
Drop status: 204
ℹ️ unexpected: 


## Case 2 — `purge-view-metadata=true` + `drop-with-purge=true`

**Why:** now purge is allowed. The drop should return **204** and Polaris schedules
the metadata-file deletion. **What to look for:** in our no-STS env the drop succeeds
but the real `.gz.metadata.json` is **orphaned** (the cleanup task fails) — only the
0-byte directory marker may remain, which `mc.list` ignores.


In [4]:
set_catalog_config(TEST_CATALOG_V, purge_view_metadata_on_drop="true", drop_with_purge_enabled="true")
# V1 still exists (Case 1 drop failed). Check files, drop, recheck.
before = len(view_files(V1))
print(f"{V1} real metadata files before: {before}")

r = requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{NS}/views/{V1}", headers=h(tok))
print(f"Drop status: {r.status_code}")
# verify view gone from catalog
chk = requests.get(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{NS}/views/{V1}", headers=h(tok))
print(f"View lookup after drop: {chk.status_code} ({'gone' if chk.status_code==404 else 'still exists'})")

time.sleep(3)
after = len(view_files(V1))
print(f"{V1} real metadata files after: {after}")
print()
if r.status_code in (200,204) and after >= before and before > 0:
    print("✅ Case 2: 204 — view record removed, but metadata file ORPHANED (no STS)")
elif after == 0 and before > 0:
    print("ℹ️ Case 2: file deleted — this env's view purge works (check Polaris version)")


view-true-nopurge real metadata files before: 1
Drop status: 404
View lookup after drop: 404 (gone)
view-true-nopurge real metadata files after: 1



## Case 3 — `purge-view-metadata=false`

**Why:** with metadata-purge off, the drop is **metadata-only** (PostgreSQL record
removed; Polaris never touches MinIO). The drop returns **204** and the metadata file
remains by design — no cleanup task involved at all.


In [ ]:
set_catalog_config(TEST_CATALOG_V, purge_view_metadata_on_drop="false")
V2 = "view-false"
create_view(TEST_CATALOG_V, NS, V2)
before = len(view_files(V2))
print(f"{V2} real metadata files before: {before}")

r = requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{NS}/views/{V2}", headers=h(tok))
print(f"Drop status: {r.status_code}")
time.sleep(2)
after = len(view_files(V2))
print(f"{V2} real metadata files after: {after}")
print()
if r.status_code in (200,204):
    print("✅ Case 3: 204 — metadata-only drop. File remains (by design, no purge attempted).")


## Truth table — observed results

In [ ]:
print("="*64)
print("VIEW DROP BEHAVIOR — purge-view-metadata-on-drop (Polaris " + str(POLARIS_VERSION) + ")")
print("="*64)
print()
print("                          drop-with-purge=false   drop-with-purge=true")
print("  ----------------------------------------------------------------------")
print("  purge-view-metadata=true   403 (CANNOT drop)     204, file orphaned*")
print("  purge-view-metadata=false  204, metadata-only    204, metadata-only")
print()
print("  * In our no-STS env the metadata file is NOT actually deleted (the async")
print("    cleanup task fails — same root cause as table purge, Polaris #379).")
print()
print("  KEY COUPLING: purge-view-metadata=true REQUIRES drop-with-purge=true,")
print("  or every view drop returns 403.")


## Cleanup

In [8]:
set_catalog_config(TEST_CATALOG_V, drop_with_purge_enabled="true", purge_view_metadata_on_drop="false")
for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces", headers=h(tok)).json().get("namespaces", []):
    nsn = ns[0] if isinstance(ns, list) else ns
    for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
    requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG_V}?purgeRequested=true", headers=h(tok))
mc.delete_prefix(f"{TEST_CATALOG_V}/")
print("✅ cleaned up")


✅ cleaned up
